<a href="https://colab.research.google.com/github/DJ-Ortega/unsupervised-trading-strat/blob/main/Unsupervised_strat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning Trading Strategy

* Download/Load SP500 stocks prices data.
* Calculate different features and indicators on each stock.
* Aggregate on monthly level and filter top 150 most liquid stocks.
* Calculate Monthly Returns for different time-horizons.
* Download Fama-French Factors and Calculate Rolling Factor Betas.
* For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.
* For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization.
* Visualize Portfolio returns and compare to SP500 returns.

Be mindful that this backtest does not take in survivorship bias

In [5]:
!pip install pandas-datareader --quiet
!pip install pyportfolioopt --quiet
!pip install pandas_ta --quiet
!pip install --upgrade yfinance --quiet
!pip install pandas==2.3.3 --quiet

In [1]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

In [17]:
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (student learning project)'}
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-30'
start_date = pd.to_datetime(end_date) - pd.DateOffset(365*8)

df = yf.download(tickers=symbols_list,
                 start = start_date,
                 end=end_date)
df

[*********************100%***********************]  503 of 503 completed


Price            Close                                                  \
Ticker               A        AAPL        ABBV        ABNB         ABT   
Date                                                                     
2018-11-01   61.491673   52.496872   57.475540         NaN   60.562435   
2018-11-02   62.161068   49.014717   57.009766         NaN   60.596954   
2018-11-05   62.142235   47.623280   59.173790         NaN   60.769634   
2018-11-06   62.208244   48.138283   59.947678         NaN   61.261734   
2018-11-07   64.357903   49.598232   62.312325         NaN   62.591236   
...                ...         ...         ...         ...         ...   
2026-10-23  173.070007  347.730011  285.640015  172.009995  103.699997   
2026-10-26  175.960007  343.910004  286.739990  171.050003  104.879997   
2026-10-27  177.490005  347.100006  286.390015  168.820007  103.099998   
2026-10-28  181.479996  343.829987  285.989990  166.044998  102.639999   
2026-10-29  181.149994  348.910004  290.790009  166.449997  104.570000   

Price                                                                  ...  \
Ticker            ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                   ...   
2018-11-01   27.281279  139.447556  245.279999   76.320778  38.520954  ...   
2018-11-02   26.653687  138.401184  239.639999   75.609596  37.951691  ...   
2018-11-05   26.463507  139.465164  239.889999   74.915779  38.030746  ...   
2018-11-06   26.511049  140.186218  240.559998   76.286072  37.421936  ...   
2018-11-07   26.777302  144.099335  253.429993   76.589645  38.188881  ...   
...                ...         ...         ...         ...        ...  ...   
2026-10-23  101.690002  249.369995  263.820007  391.070007  80.139999  ...   
2026-10-26  102.339996  247.229996  258.619995  407.929993  79.010002  ...   
2026-10-27  100.339996  235.160004  248.360001  425.350006  78.199997  ...   
2026-10-28  102.500000  243.399994  249.759995  428.269989  77.230003  ...   
2026-10-29  103.940002  247.220001  248.119995  433.279999  76.199997  ...   

Price         Volume                                                           \
Ticker            WY     WYNN      XEL       XOM       XYL       XYZ      YUM   
Date                                                                            
2018-11-01   4994500  6067500  4392500  14159300   1501000  13574000  3116400   
2018-11-02   9590300  3883300  5610000  19350400   1232600  13561600  3007400   
2018-11-05   4063400  2213300  3379300  15815000   1260700  16851400  2475400   
2018-11-06   3143800  1523300  2571600   9063200   1326900  18563300  2998200   
2018-11-07   3348000  5263100  5141900  12699800   1198400  24041900  2727500   
...              ...      ...      ...       ...       ...       ...      ...   
2026-10-23   6047700  2266800  3988700  11517700   2165000   5158100  2555300   
2026-10-26   9141900  2449100  5555900  10893300   2908400   5204700  2518200   
2026-10-27   8441300  1946700  3106100  14156500  10227600   4800900  2214600   
2026-10-28   6507300  2022100  5137400  12715300  11407100   2489300  1884900   
2026-10-29  10185000  2084200  6577800  10961300   4919900   4331600  1606700   

Price                                  
Ticker          ZBH     ZBRA      ZTS  
Date                                   
2018-11-01  1758519   521200  4277000  
2018-11-02  1135678   433400  2685300  
2018-11-05  1230644   519300  3133100  
2018-11-06  1000851  1238900  2478900  
2018-11-07  1526872   678600  2613700  
...             ...      ...      ...  
2026-10-23  2016800   451000  2860100  
2026-10-26  2681700   478900  2903900  
2026-10-27  1731600   621300  2833900  
2026-10-28  2169900   366800  2617600  
2026-10-29  2040800   527500  2935500  

[2008 rows x 2515 columns]